# VAE Latent Space Interpolation

For each saved epoch checkpoint we:
1. Encode two MNIST images from **different classes** → μ₁, μ₂
2. Sweep λ ∈ [0, 1] and decode  λμ₁ + (1−λ)μ₂
3. Visualise all epochs in a grid to see how the interpolation property emerges during training.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt
import numpy as np
from src.model import VAE
from src.data import get_mnist, get_cifar10


DATA_DICT = {
    'mnist': (1, 28, get_mnist),
    'cifar10': (3, 32, get_cifar10),
}


DATASET = 'mnist'
BETA = 1.0
DATA_CONFIG = DATA_DICT[DATASET]

In [ ]:
# ── Config (must match src/experiment/vae_train.py) ───────────────────────────
IN_CHANNELS = DATA_CONFIG[0]
INPUT_SIZE  = DATA_CONFIG[1]
HIDDEN_DIM  = 512
LATENT_DIM  = 32
NUM_HIDDEN  = 2

CKPT_DIR = ROOT / 'params' / 'vae' / DATASET / str(BETA)
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

N_STEPS   = 10   # number of λ values (endpoints included)
CLASS_A   = 1    # source class
CLASS_B   = 8    # target class
IMG_IDX_A = 0    # which image within CLASS_A
IMG_IDX_B = 0    # which image within CLASS_B

print(f'Device      : {DEVICE}')
print(f'Checkpoints : {CKPT_DIR}')

In [ ]:
# ── Pick one test image per class ─────────────────────────────────────────────
_, testset = get_mnist()
targets = torch.tensor(testset.targets)

idx_a = (targets == CLASS_A).nonzero(as_tuple=True)[0][IMG_IDX_A].item()
idx_b = (targets == CLASS_B).nonzero(as_tuple=True)[0][IMG_IDX_B].item()

x1 = testset[idx_a][0].unsqueeze(0).to(DEVICE)  # (1, 1, 28, 28)
x2 = testset[idx_b][0].unsqueeze(0).to(DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(3, 1.5))
axes[0].imshow(x1.squeeze().cpu(), cmap='gray')
axes[0].set_title(f'X₁  (class {CLASS_A})', fontsize=9)
axes[0].axis('off')
axes[1].imshow(x2.squeeze().cpu(), cmap='gray')
axes[1].set_title(f'X₂  (class {CLASS_B})', fontsize=9)
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def load_vae(ckpt_path):
    model = VAE(
        in_channels=IN_CHANNELS,
        input_size=INPUT_SIZE,
        hidden_dim=HIDDEN_DIM,
        latent_dim=LATENT_DIM,
        num_hidden=NUM_HIDDEN,
    ).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    return model


@torch.no_grad()
def interpolate(model, x1, x2, n_steps):
    """Encode with μ (no reparameterisation), sweep λ, decode.
    Returns:
        frames : tensor (n_steps, C, H, W)  – decoded latent interpolation
        mses   : list[float]                – MSE(λX1+(1-λ)X2, f(λZ1+(1-λ)Z2)) per step
    """
    mu1, _ = model.encode(x1)   # deterministic: use μ directly
    mu2, _ = model.encode(x2)
    lambdas = torch.linspace(0, 1, n_steps, device=DEVICE)
    frames, mses = [], []
    for lam in lambdas:
        z      = lam * mu1 + (1 - lam) * mu2
        recon  = model.decode(z).view(1, IN_CHANNELS, INPUT_SIZE, INPUT_SIZE)
        x_mix  = lam * x1 + (1 - lam) * x2          # pixel-space interpolant
        mse    = ((recon - x_mix) ** 2).mean().item()
        frames.append(recon.cpu())
        mses.append(mse)
    return torch.cat(frames, dim=0), mses

In [ ]:
# ── Run interpolation for every epoch checkpoint ───────────────────────────────
ckpts = sorted(CKPT_DIR.glob('epoch*.pt'))
print(f'Found {len(ckpts)} checkpoints')

epoch_labels = []
all_frames   = []
all_mses     = []

for ckpt in ckpts:
    epoch_num = int(ckpt.stem.replace('epoch', ''))
    model  = load_vae(ckpt)
    frames, mses = interpolate(model, x1, x2, N_STEPS)
    all_frames.append(frames)
    all_mses.append(mses)
    epoch_labels.append(epoch_num)

print('Done.')

In [ ]:
# ── Visualise: rows = epochs, columns = λ ─────────────────────────────────────
n_epochs = len(all_frames)
lambdas  = torch.linspace(0, 1, N_STEPS, device=DEVICE)
lambdas_display = lambdas.cpu().numpy()

# groundtruth pixel-space interpolation (shared across all rows)
gt_frames = [(lam * x1 + (1 - lam) * x2).squeeze().cpu() for lam in lambdas]

fig, axes = plt.subplots(
    n_epochs + 1, N_STEPS,
    figsize=(N_STEPS * 1.2, (n_epochs + 1) * 1.4),
    gridspec_kw={'hspace': 0.3, 'wspace': 0.05},
)

# ── Row 0: ground truth λX₁ + (1-λ)X₂ ───────────────────────────────────────
for col in range(N_STEPS):
    ax = axes[0, col]
    ax.imshow(gt_frames[col], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f'λ={lambdas_display[col]:.1f}', fontsize=10)
axes[0, 0].set_ylabel('ground\ntruth', fontsize=7, rotation=0, labelpad=28, va='center')

# ── Rows 1+: decoded latent interpolation per epoch ──────────────────────────
for row, (frames, mses, epoch) in enumerate(zip(all_frames, all_mses, epoch_labels)):
    for col in range(N_STEPS):
        ax = axes[row + 1, col]
        ax.imshow(frames[col].squeeze(), cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        ax.text(0.5, -0.04, f'MSE={mses[col]:.4f}',
                ha='center', va='top', fontsize=5,
                transform=ax.transAxes)
    axes[row + 1, 0].set_ylabel(f'ep {epoch}', fontsize=7, rotation=0, labelpad=28, va='center')

fig.suptitle(
    f'VAE latent interpolation  |  class {CLASS_A} (λ=1)  →  class {CLASS_B} (λ=0)',
    fontsize=20, y=0.98, weight='bold'
)

out_path = ROOT / 'result' / 'vae' / DATASET / str(BETA) / 'interpolation.png'
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.tight_layout()
plt.show()
print(f'Saved → {out_path.relative_to(ROOT)}')

## Observations

| Epoch range | Expected behaviour |
|---|---|
| **Early** | Blurry / noisy midpoints — latent space not yet organised |
| **Mid** | Gradual transition begins to appear; midpoints may look ambiguous |
| **Late** | Smooth morphing; intermediate λ values produce legible in-between digits |

The KL regularisation term forces the encoder toward a standard-Gaussian posterior, making the latent space *convex*: any linear interpolation between two codes stays in a high-density region, which is exactly what enables smooth interpolation.

# Interpolation across β values

For each β, load the **best** epoch checkpoint and show:
- **Left label**: β, best-epoch eval loss, and target linearity (TL) at that epoch
- **Columns**: decoded frames sweeping λ ∈ [0, 1]

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt
import numpy as np
from src.model import VAE
from src.data import get_mnist, get_cifar10
from src.utils import load_json

DATA_DICT = {
    'mnist': (1, 28, get_mnist),
    'cifar10': (3, 32, get_cifar10),
}

DATASET     = 'mnist'
DATA_CONFIG = DATA_DICT[DATASET]
IN_CHANNELS = DATA_CONFIG[0]
INPUT_SIZE  = DATA_CONFIG[1]
HIDDEN_DIM  = 512
LATENT_DIM  = 32
NUM_HIDDEN  = 2

N_STEPS   = 10
CLASS_A   = 8
CLASS_B   = 2
IMG_IDX_A = 0
IMG_IDX_B = 0

PARAMS_ROOT  = ROOT / 'params' / 'vae' / DATASET
RESULTS_ROOT = ROOT / 'result' / 'vae' / DATASET
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

In [ ]:
# ── Pick test images & define helpers ─────────────────────────────────────────
_, testset = DATA_CONFIG[2]()
targets = torch.tensor(testset.targets)

idx_a = (targets == CLASS_A).nonzero(as_tuple=True)[0][IMG_IDX_A].item()
idx_b = (targets == CLASS_B).nonzero(as_tuple=True)[0][IMG_IDX_B].item()

x1 = testset[idx_a][0].unsqueeze(0).to(DEVICE)
x2 = testset[idx_b][0].unsqueeze(0).to(DEVICE)


def load_vae(ckpt_path):
    model = VAE(
        in_channels=IN_CHANNELS,
        input_size=INPUT_SIZE,
        hidden_dim=HIDDEN_DIM,
        latent_dim=LATENT_DIM,
        num_hidden=NUM_HIDDEN,
    ).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    return model


@torch.no_grad()
def interpolate(model, x1, x2, n_steps):
    """Returns frames (n_steps, C, H, W) and MSE(λX1+(1-λ)X2, f(λZ1+(1-λ)Z2)) per step."""
    mu1, _ = model.encode(x1)
    mu2, _ = model.encode(x2)
    lambdas = torch.linspace(0, 1, n_steps, device=DEVICE)
    frames, mses = [], []
    for lam in lambdas:
        z     = lam * mu1 + (1 - lam) * mu2
        recon = model.decode(z).view(1, IN_CHANNELS, INPUT_SIZE, INPUT_SIZE)
        x_mix = lam * x1 + (1 - lam) * x2
        mse   = ((recon - x_mix) ** 2).mean().item()
        frames.append(recon.cpu())
        mses.append(mse)
    return torch.cat(frames, dim=0), mses

In [ ]:
# ── Collect all available β values ────────────────────────────────────────────
beta_dirs = sorted(PARAMS_ROOT.iterdir(), key=lambda p: float(p.name))
betas = [float(p.name) for p in beta_dirs]
print(f'Found β values: {betas}')

# ── Per-β: find best checkpoint, load metrics ─────────────────────────────────
rows = []   # list of dicts with keys: beta, best_epoch, frames, mses, eval_loss, tl

for beta_dir in beta_dirs:
    beta = float(beta_dir.name)

    best_ckpts = list(beta_dir.glob('best_*.pt'))
    if not best_ckpts:
        print(f'  β={beta}: no best checkpoint, skipping')
        continue
    best_ckpt  = best_ckpts[0]
    best_epoch = int(best_ckpt.stem.split('_')[1])   # "best_10" → 10

    result_dir = RESULTS_ROOT / beta_dir.name
    tl_data    = load_json(result_dir / 'tl.json')
    loss_data  = load_json(result_dir / 'loss.json')

    tl_val     = tl_data['target_linearity'][best_epoch - 1]
    loss_entry = next(d for d in loss_data if d['epoch'] == best_epoch)
    eval_loss  = loss_entry['eval_loss']

    model        = load_vae(best_ckpt)
    frames, mses = interpolate(model, x1, x2, N_STEPS)

    rows.append(dict(beta=beta, best_epoch=best_epoch,
                     frames=frames, mses=mses, eval_loss=eval_loss, tl=tl_val))
    print(f'  β={beta:5.2f}  best_epoch={best_epoch:3d}  '
          f'eval_loss={eval_loss:.2f}  TL={tl_val:.4f}')

print('Done.')

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────────
def to_img(t):
    """Convert a (C, H, W) or (1, C, H, W) tensor to a numpy array for imshow.
    Grayscale (C=1) → (H, W);  RGB (C=3) → (H, W, 3), values in [0, 1].
    """
    t = t.squeeze()          # (H, W) or (C, H, W)
    if t.ndim == 3:          # RGB
        t = t.permute(1, 2, 0).clamp(0, 1)
    return t.cpu().numpy()

def show(ax, t, **kwargs):
    img = to_img(t)
    cmap = 'gray' if img.ndim == 2 else None
    ax.imshow(img, cmap=cmap, vmin=0, vmax=1, **kwargs)

# ── Visualise: rows = β values, columns = λ ───────────────────────────────────
n_rows = len(rows)
lambdas  = torch.linspace(0, 1, N_STEPS, device=DEVICE)
lambdas_display = lambdas.cpu().numpy()

# groundtruth pixel-space interpolation (same for all β)
gt_frames = [(lam * x1 + (1 - lam) * x2).squeeze(0).cpu() for lam in lambdas]

n_cols = N_STEPS + 1
col_w  = [2.0] + [1.2] * N_STEPS

fig, axes = plt.subplots(
    n_rows + 1, n_cols,
    figsize=(sum(col_w), (n_rows + 1) * 1.5),
    gridspec_kw={'width_ratios': col_w, 'hspace': 0.35, 'wspace': 0.04},
)

if n_rows + 1 == 1:
    axes = axes[np.newaxis, :]

# ── Row 0: ground truth λX₁ + (1-λ)X₂ ───────────────────────────────────────
ax_txt = axes[0, 0]
ax_txt.axis('off')
# ax_txt.set_title('metrics', fontsize=9)
ax_txt.text(0.5, 0.5, 'Ground\nTruth',
            ha='center', va='center', fontsize=12,
            transform=ax_txt.transAxes,
                fontweight='bold')

for col in range(N_STEPS):
    ax = axes[0, col + 1]
    show(ax, gt_frames[col])
    ax.axis('off')
    ax.set_title(f'$\\alpha$={lambdas_display[col]:.1f}', fontsize=12)

# ── Rows 1+: one row per β ────────────────────────────────────────────────────
for row, r in enumerate(rows):
    ax_txt = axes[row + 1, 0]
    ax_txt.axis('off')
    label = (
        f"β = {r['beta']}\n"
        f"TL {r['tl']:.4f}"
    )
    ax_txt.text(0.5, 0.5, label,
                ha='center', va='center', fontsize=12,
                fontweight='bold',
                transform=ax_txt.transAxes)

    for col in range(N_STEPS):
        ax = axes[row + 1, col + 1]
        show(ax, r['frames'][col])
        ax.axis('off')
        # ax.text(0.5, -0.04, f'MSE={r["mses"][col]:.4f}',
        #         ha='center', va='top', fontsize=5,
        #         transform=ax.transAxes)

fig.suptitle(
    f'VAE interpolation across β',
    fontsize=18, y=0.95, weight='bold',
)

out_path = ROOT / 'plot' / 'vae_interpolation.png'
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.tight_layout()
plt.show()
print(f'Saved → {out_path.relative_to(ROOT)}')